In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\jagad\Downloads\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory) 



    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf' 

            all_documents.extend(documents)
            print(f" loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal Documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../Data")



Found 2 PDF files to process

 processing: Machine_Learning_Lecture.pdf
 loaded 435 pages

 processing: mml-book.pdf
 loaded 417 pages

Total Documents loaded: 852


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-28T07:35:12-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev', 'source': '..\\Data\\pdf\\Machine_Learning_Lecture.pdf', 'total_pages': 435, 'page': 0, 'page_label': 'i', 'source_file': 'Machine_Learning_Lecture.pdf', 'file_type': 'pdf'}, page_content='Mathematical Foundations\nof Machine Learning\nLectures on YouTube:\nhttps://www.youtube.com/@mathtalent\nSeongjai Kim\nDepartment of Mathematics and Statistics\nMississippi State University\nMississippi State, MS 39762 USA\nEmail: skim@math.msstate.edu\nUpdated: April 28, 2025'),
 Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', '

In [8]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")


    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}.....")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs




In [9]:
chunks = split_documents(all_pdf_documents)
chunks

Split 852 documents into 1907 chunks

Example Chunk:
Content: Mathematical Foundations
of Machine Learning
Lectures on YouTube:
https://www.youtube.com/@mathtalent
Seongjai Kim
Department of Mathematics and Statistics
Mississippi State University
Mississippi Sta.....
Metadata: {'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-28T07:35:12-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev', 'source': '..\\Data\\pdf\\Machine_Learning_Lecture.pdf', 'total_pages': 435, 'page': 0, 'page_label': 'i', 'source_file': 'Machine_Learning_Lecture.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-28T07:35:12-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev', 'source': '..\\Data\\pdf\\Machine_Learning_Lecture.pdf', 'total_pages': 435, 'page': 0, 'page_label': 'i', 'source_file': 'Machine_Learning_Lecture.pdf', 'file_type': 'pdf'}, page_content='Mathematical Foundations\nof Machine Learning\nLectures on YouTube:\nhttps://www.youtube.com/@mathtalent\nSeongjai Kim\nDepartment of Mathematics and Statistics\nMississippi State University\nMississippi State, MS 39762 USA\nEmail: skim@math.msstate.edu\nUpdated: April 28, 2025'),
 Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', '

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [11]:
from chromadb.config import Settings

In [12]:
class EmbeddingManager:

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):

        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded succesfully. Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating Embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar = True)
        print(f"Generate Embediings with Shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()

embedding_manager



Loading Embedding model: all-MiniLM-L6-v2
Model Loaded succesfully. Embedding Dimension : 384


In [13]:
#### vector Store

class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../Data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description" : "PDF document embeddings for RAG"}
            )

            print(f" vector Store initialized. collection : {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:

            print(f"error initializing vector store: {e}")
            raise
        

    def add_documents( self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must watch number of embeddings")
        
        print(f"Adding {len(documents)} documents to the vector store...")
              
        
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)


            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)


            document_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = document_text
            )

            print(f"sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")


        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


VectorStore = VectorStore()
VectorStore
        


    


 vector Store initialized. collection : pdf_documents
Existing document in collection: 7628


In [14]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-28T07:35:12-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev', 'source': '..\\Data\\pdf\\Machine_Learning_Lecture.pdf', 'total_pages': 435, 'page': 0, 'page_label': 'i', 'source_file': 'Machine_Learning_Lecture.pdf', 'file_type': 'pdf'}, page_content='Mathematical Foundations\nof Machine Learning\nLectures on YouTube:\nhttps://www.youtube.com/@mathtalent\nSeongjai Kim\nDepartment of Mathematics and Statistics\nMississippi State University\nMississippi State, MS 39762 USA\nEmail: skim@math.msstate.edu\nUpdated: April 28, 2025'),
 Document(metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-28T07:35:12-05:00', 'author': '', 'title': '', '

In [15]:
texts = [doc.page_content for doc in chunks]

texts

['Mathematical Foundations\nof Machine Learning\nLectures on YouTube:\nhttps://www.youtube.com/@mathtalent\nSeongjai Kim\nDepartment of Mathematics and Statistics\nMississippi State University\nMississippi State, MS 39762 USA\nEmail: skim@math.msstate.edu\nUpdated: April 28, 2025',
 'Seongjai Kim, Professor of Mathematics, Department of Mathematics and Statistics, Mississippi\nState University, Mississippi State, MS 39762 USA. Email: skim@math.msstate.edu.',
 'Prologue\nIn organizing this lecture note, I am indebted by the following:\n• S. RASCHKA AND V. MIRJALILI , Python Machine Learning, 3rd Ed., 2019\n[62].\n• (Lecture note) http://fa.bianp.net/teaching/2018/eecs227at/, Dr. Fabian Pe-\ndregosa, UC Berkeley\n• (Lecture note) Introduction To Machine Learning, Prof. David Sontag,\nMIT & NYU\n• (Lecture note) Mathematical Foundations of Machine Learning, Dr. Justin\nRomberg, Geoigia Tech\nThis lecture note will grow up as time marches; various core algorithms,\nuseful techniques, and i

In [16]:
embeddings = embedding_manager.generate_embeddings(texts)


VectorStore.add_documents(chunks, embeddings)

Generating Embeddings for 1907 texts...


Batches: 100%|██████████| 60/60 [01:17<00:00,  1.29s/it]


Generate Embediings with Shape: (1907, 384)
Adding 1907 documents to the vector store...
sucessfully added 1907 documents to vector store
Total documents in collection: 9535


In [17]:
###Retriever Pipeline From VectorStore


In [21]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int =5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top_k : {top_k}, score threshold: {score_threshold}")


        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1- distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content' : document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance' : distance,
                            'rank': i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:

                print(f"no documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"errors during retreival: {e}")
            return []
        
rag_retriever = RAGRetriever(VectorStore, embedding_manager)


In [22]:



rag_retriever

In [23]:
rag_retriever.retrieve("what is machine learning")

Retrieving documents for query: what is machine learning
Top_k : 5, score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.77it/s]

Generate Embediings with Shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8bbd113c_714',
  'content': 'Foreword\nMachine learning is the latest in a long line of attempts to distill human\nknowledge and reasoning into a form that is suitable for constructing ma-\nchines and engineering automated systems. As machine learning becomes\nmore ubiquitous and its software packages become easier to use, it is nat-\nural and desirable that the low-level technical details are abstracted away\nand hidden from the practitioner. However, this brings with it the danger\nthat a practitioner becomes unaware of the design decisions and, hence,\nthe limits of machine learning algorithms.\nThe enthusiastic practitioner who is interested to learn more about the\nmagic behind successful machine learning algorithms currently faces a\ndaunting set of pre-requisite knowledge:\nProgramming languages and data analysis tools\nLarge-scale computation and the associated frameworks\nMathematics and statistics and how machine learning builds on it\nAt universities, introducto

In [30]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


False

In [32]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


True

In [37]:
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.1-8b-instant", temperature = 0.1, max_tokens = 1024)


In [38]:
def rag_simple(query, retriever, llm, top_k = 3):

    results = retriever.retrieve(query, top_k =3)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No revelant context found to answer the question."
    
    prompt = f"""Use the following context tp answer thr question concisely.

        Context:
        {context}
        Question: {query}
        
        Answer:

    """
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [39]:
answer = rag_simple("what is machine learning", rag_retriever, llm)
print(answer)

Retrieving documents for query: what is machine learning
Top_k : 3, score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.91it/s]

Generate Embediings with Shape: (1, 384)
Retrieved 3 documents (after filtering)


Machine learning is the latest in a long line of attempts to distill human knowledge and reasoning into a form that is suitable for constructing machines and engineering automated systems.


In [40]:
print(answer)

Machine learning is the latest in a long line of attempts to distill human knowledge and reasoning into a form that is suitable for constructing machines and engineering automated systems.


In [41]:
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):

    results = retriever.retrieve(query, top_k  = top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence' : 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""use the following context to answer the question correctly. \n Context: \n{context} \n\n Questions: {query} \n\n Answer """


    response = llm.invoke([prompt.format(context = context, query = query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context 
    return output


In [42]:
result = rag_advanced("what is machine learning", rag_retriever, llm, top_k = 3, min_score= 0.1, return_context = True)
print("answer:", result['answer'])
print("sources:", result['sources'])
print("confidence:", result['confidence'])
print("context preview:", result['context'][:300])

Retrieving documents for query: what is machine learning
Top_k : 3, score threshold: 0.1
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.45it/s]

Generate Embediings with Shape: (1, 384)
Retrieved 3 documents (after filtering)


answer: Machine learning is the latest in a long line of attempts to distill human knowledge and reasoning into a form that is suitable for constructing machines and engineering automated systems.
sources: [{'source': 'mml-book.pdf', 'page': 6, 'score': 0.3859015703201294, 'preview': 'Foreword\nMachine learning is the latest in a long line of attempts to distill human\nknowledge and reasoning into a form that is suitable for constructing ma-\nchines and engineering automated systems. As machine learning becomes\nmore ubiquitous and its software packages become easier to use, it is na...'}, {'source': 'mml-book.pdf', 'page': 6, 'score': 0.3859015703201294, 'preview': 'Foreword\nMachine learning is the latest in a long line of attempts to distill human\nknowledge and reasoning into a form that is suitable for constructing ma-\nchines and engineering automated systems. As machine learning becomes\nmore ubiquitous and its software packages become easier to use, it is na...'}, {'source': 'm